In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
PROJECT_DIR = Path.cwd().parent

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

WEATHER_FILE = (
    PROCESSED_DIR /
    "weather_daily_era5_cleaned.csv"
)

print("Weather file:", WEATHER_FILE)

Weather file: c:\Users\Nethmi\Documents\GitHub\RiskWeave\data\processed\weather_daily_era5_cleaned.csv


In [3]:
weather = pd.read_csv(WEATHER_FILE)

weather["date"] = pd.to_datetime(weather["date"])

print("Dataset shape:", weather.shape)
print("Number of districts:", weather["district"].nunique())
print("First date:", weather["date"].min())
print("Last date:", weather["date"].max())

weather.head()

Dataset shape: (16775, 14)
Number of districts: 25
First date: 2024-12-30 00:00:00
Last date: 2026-10-31 00:00:00


,district,date,precipitation_sum,temperature_2m_mean,temperature_2m_max,temperature_2m_min,representative_latitude,representative_longitude,api_grid_latitude,api_grid_longitude,api_grid_elevation_m,weather_model,weather_source,timezone
0,Ampara,2024-12-30,5.6,25.9,28.5,23.6,7.124905,81.685472,7.0,81.75,48.0,era5,Open-Meteo Historical Weather API,Asia/Colombo
1,Ampara,2024-12-31,16.1,25.6,27.6,24.4,7.124905,81.685472,7.0,81.75,48.0,era5,Open-Meteo Historical Weather API,Asia/Colombo
2,Ampara,2025-01-01,26.5,25.1,26.7,24.3,7.124905,81.685472,7.0,81.75,48.0,era5,Open-Meteo Historical Weather API,Asia/Colombo
3,Ampara,2025-01-02,9.9,24.8,25.4,24.2,7.124905,81.685472,7.0,81.75,48.0,era5,Open-Meteo Historical Weather API,Asia/Colombo
4,Ampara,2025-01-03,1.7,25.8,28.2,24.1,7.124905,81.685472,7.0,81.75,48.0,era5,Open-Meteo Historical Weather API,Asia/Colombo


In [4]:
weather.groupby("district").agg(
    total_records=("date", "count"),
    unique_dates=("date", "nunique"),
    first_date=("date", "min"),
    last_date=("date", "max")
)

,total_records,unique_dates,first_date,last_date
district,,,,
Ampara,671,671,2024-12-30,2026-10-31
Anuradhapura,671,671,2024-12-30,2026-10-31
Badulla,671,671,2024-12-30,2026-10-31
Batticaloa,671,671,2024-12-30,2026-10-31
Colombo,671,671,2024-12-30,2026-10-31
Galle,671,671,2024-12-30,2026-10-31
Gampaha,671,671,2024-12-30,2026-10-31
Hambantota,671,671,2024-12-30,2026-10-31
Jaffna,671,671,2024-12-30,2026-10-31


In [5]:
reporting_anchors = {
    2025: pd.Timestamp("2024-12-30"),
    2026: pd.Timestamp("2025-12-29")
}

for year, anchor in reporting_anchors.items():
    print(
        f"{year} Week 01: "
        f"{anchor.date()} to "
        f"{(anchor + pd.Timedelta(days=6)).date()}"
    )

2025 Week 01: 2024-12-30 to 2025-01-05
2026 Week 01: 2025-12-29 to 2026-01-04


In [8]:
calendar_records = []

for year, anchor in reporting_anchors.items():

    next_anchor = reporting_anchors.get(year + 1)

    if next_anchor is not None:
        period_end = next_anchor - pd.Timedelta(days=1)
    else:
        period_end = weather["date"].max()

    week_start = anchor
    week_number = 1

    while week_start <= period_end:

        week_end = week_start + pd.Timedelta(days=6)

        calendar_records.append({
            "reporting_year": year,
            "week_number": week_number,
            "week_start": week_start,
            "week_end": week_end
        })

        week_start += pd.Timedelta(days=7)
        week_number += 1

weekly_calendar = pd.DataFrame(calendar_records)

weekly_calendar.head(10)

,reporting_year,week_number,week_start,week_end
0,2025,1,2024-12-30,2025-01-05
1,2025,2,2025-01-06,2025-01-12
2,2025,3,2025-01-13,2025-01-19
3,2025,4,2025-01-20,2025-01-26
4,2025,5,2025-01-27,2025-02-02
5,2025,6,2025-02-03,2025-02-09
6,2025,7,2025-02-10,2025-02-16
7,2025,8,2025-02-17,2025-02-23
8,2025,9,2025-02-24,2025-03-02
9,2025,10,2025-03-03,2025-03-09
